# 1. Text exploration — MulTaBench core text

Audit the **20 official core TEXT datasets** before any Jev call. Focus on text-column counts, lengths, missing inputs, exact duplicates and the amount of reusable extraction. All source versions and text-column lists are pinned in `config/exploration/datasets.yaml`.

## 1. Scope and provenance

The public CSVs and metadata are loaded at pinned versions. Missing data can be downloaded when `allow_download: true`; existing data are hash-checked. No model code or weights are loaded. See `docs/DATA_SOURCES.md` for the official text definition and audit denominators. Each notebook runs independently.


In [ ]:
import sys
from pathlib import Path
REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.utils.config import load_config, save_resolved_config
from src.utils.notebook_display import display_local, display_json
from src.utils.provenance import provenance
from src.data.audit import audit_collection
from src.visualize import figures, style
style.apply()
cfg = load_config()
from src.data.audit import audit_summary
from src.visualize.audit import burden_figures, reuse_figure
import matplotlib.pyplot as plt

NOTEBOOK = "01_data_exploration"
from src.utils.notebook_report import begin_report, report_section, finish_report
begin_report()
report_section('1. Scope and provenance')
save = figures.FigureSaver(NOTEBOOK)
config_hash = save_resolved_config(cfg, NOTEBOOK)
display_json("Execution provenance", provenance())
audit = audit_collection(cfg)


## 2. Dataset overview

One row per dataset. Feature counts exclude the target. Binary versus multiclass is determined from the official classification task and observed distinct class vocabulary; numerical targets are not automatically treated as regression. Class vocabulary may be inspected below; no frequencies enter request metadata.

In [ ]:
report_section('2. Dataset overview')
display_local(audit.summary[["dataset", "rows", "features", "task_type", "target", "target_dtype", "n_classes", "text_columns", "non_text_columns"]])
for dataset_id in audit.summary.dataset_id:
    display_json(audit.metadata(dataset_id).dataset_name + " — task metadata", audit.details[dataset_id]["task_metadata"])


## 3. Individual text columns: length, missingness and repetition

One row per official text column. `nonempty_characters_mean` excludes null/whitespace-only inputs; the existing character/token statistics include them as zero. `unique_nonempty_inputs` counts the exact inputs that need a first response for this field. `repeated_nonempty_rows` counts the additional copies that can reuse it; `top_input_frequencies` lists the frequencies of the most common inputs without exposing their text.

Equality is exact: nonempty whitespace, capitalization, value type, and column names matter. Null, nonfinite and whitespace-only inputs share the agreed missing policy. The detailed audit below retains descriptive dtype/target statistics separately.


In [ ]:
report_section('3. Individual text columns: length, missingness and repetition')
display_local(audit.text[["dataset_id", "feature", "nonempty_characters_mean", "characters_median", "characters_p95", "tokens_mean", "tokens_p95", "missing_pct", "missing_or_empty_pct", "nonempty_rows", "unique_nonempty_inputs", "unique_nonempty_ratio", "repeated_nonempty_rows", "top_input_frequencies"]])


## 4. Joint duplicates and total reusable extraction

Joint equality compares **all named available text fields together**, omitting missing fields. Count this directly; do not assume every joint row is unique. `joint_single_field_unique_overlap` counts joint inputs that equal a per-column input because only one field is available.

`combined_unique_inputs = per_column_unique_inputs + joint_unique_inputs - joint_single_field_unique_overlap`.

These are exact counts for fixed task metadata, question, model and configuration. They describe requests to obtain and cache once, not the number of numeric feature columns. A stored first response is reused; the API is not assumed to return identical values on repeated inference. No target values enter the matching.


In [ ]:
report_section('4. Joint duplicates and total reusable extraction')
display_local(audit.summary[["dataset", "rows", "text_columns", "per_column_nonempty_inputs", "per_column_unique_inputs", "per_column_reuse_savings_pct", "joint_nonempty_inputs", "joint_unique_inputs", "joint_unique_ratio", "joint_reuse_savings_pct", "joint_missing_or_empty_pct", "joint_single_field_unique_overlap", "combined_unique_inputs"]])
display_local(audit.reuse[audit.reuse["mode"] == "joint"])
fig, name, caption = reuse_figure(audit.summary)
save(fig, name, caption=caption)
display_local(fig)
plt.close(fig)


## 5. Full column and target audit

Expand a dataset to see **every feature**, missingness, unique counts/ratios, numeric statistics, categorical cardinalities/common values, text-length statistics and representative examples. Unique ratios use nonmissing rows as denominator. Character/word/token length statistics include missing text as empty, matching workload accounting. “Words” means whitespace-separated strings.

Tokens are approximated as `ceil(characters / 4)`, not measured with Jev’s tokenizer. Examples are unique nonempty values at deterministic length ranks near 10%, 50% and 90%, clipped only for display. Full request text is never clipped.

The target distribution or regression quantiles are **audit-only**. The request builder has no input for these statistics. “Other” includes date-like columns. Non-text feature classification uses the CSV dtype; no model preprocessing has been selected.

In [ ]:
report_section('5. Full column and target audit')
for dataset_id in audit.summary.dataset_id:
    detail = audit.details[dataset_id]
    display_json(audit.metadata(dataset_id).dataset_name + " — full column audit", {
        "features": audit.columns[audit.columns.dataset_id == dataset_id].to_dict(orient="records"),
        "text_statistics": audit.text[audit.text.dataset_id == dataset_id].to_dict(orient="records"),
        "numeric_statistics": audit.numeric[audit.numeric.dataset_id == dataset_id].fillna("missing").to_dict(orient="records"),
        "categorical_statistics": audit.categorical[audit.categorical.dataset_id == dataset_id].to_dict(orient="records"),
        "text_examples": detail["text_examples"],
        "target_audit_only": detail["target_audit_only"],
        "issues": detail["issues"],
    })


## 6. Text burden

Combined row tokens use `ceil(sum(text characters) / 4)`. Notebook 2 adds metadata, question wording, class choices and JSON escaping. No non-text features enter Jev. Short strings and names count as text under the official annotation.


In [ ]:
report_section('6. Text burden')
display_local(audit.summary[["dataset", "text_columns", "combined_tokens_mean", "combined_tokens_median", "combined_tokens_p95", "combined_tokens_max", "total_text_tokens"]])
for fig, name, caption in burden_figures(audit.summary, cfg["token_estimation"]["characters_per_token"]):
    save(fig, name, caption=caption)
    display_local(fig)
    plt.close(fig)


## 7. Saved outputs

CSV/JSON audits and encoded length arrays are under `output_JEVPFN/exploration/results/data_audit/`. Configurations are in that phase's `manifests/`. Figures are in `output_JEVPFN/figures/exploration/01_data_exploration/`, with `Captions.md`. The full final summary contains every displayed audit detail and plotted value, including local examples. Generated reports are gitignored; source notebooks remain output-free for GitHub.


In [ ]:
report_section('7. Saved outputs')
display_json("Audit artifacts", {"directory": str(audit.folder), "tables": ["summary.csv", "columns.csv", "text.csv", "reuse.csv", "numeric.csv", "categorical.csv"], "config_sha256": config_hash})


## Summary

Section order matches the notebook. The notebook saves this complete text verbatim into `output_JEVPFN/Allresults.md`.

In [ ]:
finish_report(NOTEBOOK, audit_summary(audit) + "\n" + save.summary())
